# Reading the results

This notebook loads the published runs. It does not call a model. The numbers it prints are the automated scores from [docs/results.md](../docs/results.md). Human review is a separate file and is not applied here, on purpose: the headline table was not rewritten to match the reviewer.

Run it from the repository root, or from `notebooks/`. The first cell finds the root either way.


In [ ]:
from pathlib import Path

from prompt_injection_eval.metrics import by_model_condition, paired_by_model
from prompt_injection_eval.results_io import load_results

ROOT = Path.cwd()
if not (ROOT / "results-published").is_dir():
    ROOT = ROOT.parent

rows = []
for name in ("deepseek-r1-14b", "qwen2.5-7b"):
    loaded, malformed = load_results(ROOT / "results-published" / name / "raw_results.jsonl")
    assert not malformed
    rows.extend(loaded)
print(f"{len(rows)} published units, models {sorted({r['model'] for r in rows})}")

## Headline table

Attack rates use attack cases only. Task rates use every case. Wilson intervals are binomial uncertainty for that count. They are not a test that one condition beat another.


In [ ]:
def pct(rate):
    return "n/a" if rate is None else f"{rate:.3f}"


print(f"{'model':16} {'condition':10} {'attack':18} {'wilson':18} {'useful':8} {'task':8}")
for group in by_model_condition(rows):
    ci = group["attack_success_rate_valid_ci"]
    wilson = f"[{ci['low']:.3f}, {ci['high']:.3f}]"
    attack = f"{group['attack_successes']}/{group['attack_units_valid']} {pct(group['attack_success_rate_valid'])}"
    task = f"{group['task_successes']}/{group['task_units_valid']}"
    print(
        f"{group['model']:16} {group['condition']:10} {attack:18} {wilson:18} {pct(group['secure_and_useful_rate_valid']):8} {task:8}"
    )

## Paired defense effect

Each defense is paired with baseline inside one model. Incomparable pairs, mostly ambiguous scores, are counted and then left out of the rates.


In [ ]:
for item in paired_by_model(rows):
    print(
        item["model"],
        item["defense_condition"],
        f"improved {item['improved']}",
        f"worsened {item['worsened']}",
        f"unchanged {item['unchanged']}",
        f"incomparable {item['incomparable']}",
        f"attack {pct(item['baseline_attack_success_rate'])} -> {pct(item['defended_attack_success_rate'])}",
        f"task {pct(item['baseline_task_success_rate'])} -> {pct(item['defended_task_success_rate'])}",
    )

## The comparison figure

The chart committed with the study is the same grouping as the table: attack success by model and condition, with Wilson error bars. The next cell displays that file. If you are reading the notebook on the site rather than running it, the same image is on the results page and under `results-published/compare/`.


In [ ]:
from IPython.display import Image, display

display(
    Image(filename=str(ROOT / "results-published" / "compare" / "model_condition_comparison.png"))
)
display(Image(filename=str(ROOT / "results-published" / "compare" / "model_condition_utility.png")))

## Exercise: one category

`direct_override` is four cases. Filter to it and print attack success by model and condition. You should be able to see whether the headline is carried by this category or by the rest of the file. Change `CATEGORY` and run the cell again.


In [ ]:
CATEGORY = "direct_override"
subset = [row for row in rows if row["category"] == CATEGORY]
print(f"{CATEGORY}: {len(subset)} units")
for group in by_model_condition(subset):
    print(
        group["model"],
        group["condition"],
        f"attack {group['attack_successes']}/{group['attack_units_valid']}",
        f"task {group['task_successes']}/{group['task_units_valid']}",
    )